# 03_audio_deep_learning_es_from_en — Clasificación de emociones en audio (ES)

Este notebook entrena un modelo de Deep Learning para clasificar emociones a partir de características acústicas extraídas del audio en español.

Se parte del manifest generado previamente y se construye una red neuronal que aprende patrones emocionales a partir de representaciones espectrales.

**Framework:** PyTorch  
**Entrada:** `manifest_es.csv`  
**Salida:** modelo entrenado en `../models/`

## 1) Librerías y configuración

Importamos las librerías necesarias para:
- Procesamiento de audio (`librosa`, `torchaudio`)
- Manejo de tensores (`torch`)
- Creación de datasets y DataLoaders
- Métricas de evaluación (sklearn)

También se configura el uso de GPU si está disponible.

In [1]:
import os, math, time, random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchaudio
from transformers import ASTFeatureExtractor, ASTForAudioClassification, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# Para evitar problemas en Windows
os.environ["TOKENIZERS_PARALLELISM"] = "false"


DEVICE: cuda
GPU: NVIDIA GeForce RTX 3060


## 2) Rutas y parámetros principales

Se define la ruta al `manifest_es.csv` y los principales hiperparámetros:

- Frecuencia de muestreo
- Tamaño de batch
- Número de épocas
- Learning rate

Estos parámetros controlan el entrenamiento del modelo.

In [2]:
HERE = Path.cwd().resolve()

if (HERE / "data").exists() and (HERE / "models").exists():
    PROJECT_ROOT = HERE
elif (HERE.parent / "data").exists() and (HERE.parent / "models").exists():
    PROJECT_ROOT = HERE.parent
else:
    PROJECT_ROOT = None
    for p in HERE.parents:
        if (p / "data").exists() and (p / "models").exists():
            PROJECT_ROOT = p
            break
    if PROJECT_ROOT is None:
        raise RuntimeError("No encuentro PROJECT_ROOT con carpetas data/ y models/. Ajusta PROJECT_ROOT a mano.")

DATA_AUDIO = PROJECT_ROOT / "data" / "audio"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True, parents=True)

MANIFEST_ES = DATA_AUDIO / "manifest_es.csv"    # tu csv en data/audio
CKPT_EN     = MODELS_DIR / "ast_best.pt"        # tu modelo inglés

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MANIFEST_ES:", MANIFEST_ES, "| exists:", MANIFEST_ES.exists())
print("CKPT_EN:", CKPT_EN, "| exists:", CKPT_EN.exists())

if not MANIFEST_ES.exists():
    raise FileNotFoundError(f"No existe {MANIFEST_ES}. Revisa que esté en data/audio/manifest_es.csv")


PROJECT_ROOT: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion
MANIFEST_ES: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\manifest_es.csv | exists: True
CKPT_EN: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\models\ast_best.pt | exists: True


In [3]:
CFG = {
    "model_name": "MIT/ast-finetuned-audioset-10-10-0.4593",
    "sampling_rate": 16000,

    # Entrenamiento
    "epochs": 25,
    "lr": 2e-5,
    "lr_finetune": 2e-6,
    "weight_decay": 0.02,
    "warmup_ratio": 0.10,
    "label_smoothing": 0.05,

    # Ajusta para exprimir GPU sin petar VRAM
    "batch_size": 24,
    "grad_accum": 2,  # batch efectivo = 48

    # 🔥 CLAVE anti-cuelgues en Windows:
    "num_workers": 0,
    "pin_memory": False,
    "persistent_workers": False,

    # clipping
    "clip_norm": 1.0,

    # early stopping
    "patience": 5,

    # Augment (suave)
    "specaugment": True,
    "time_mask_param": 60,
    "freq_mask_param": 18,
}
print(CFG)


{'model_name': 'MIT/ast-finetuned-audioset-10-10-0.4593', 'sampling_rate': 16000, 'epochs': 25, 'lr': 2e-05, 'lr_finetune': 2e-06, 'weight_decay': 0.02, 'warmup_ratio': 0.1, 'label_smoothing': 0.05, 'batch_size': 24, 'grad_accum': 2, 'num_workers': 0, 'pin_memory': False, 'persistent_workers': False, 'clip_norm': 1.0, 'patience': 5, 'specaugment': True, 'time_mask_param': 60, 'freq_mask_param': 18}


## 3) Cargar manifest y preparar etiquetas

Se carga el manifest con las rutas de audio y sus etiquetas de emoción.  
Se convierten las etiquetas a formato numérico para poder entrenar el modelo.

In [4]:
df = pd.read_csv(MANIFEST_ES)
print("Rows:", len(df))
print("Cols:", df.columns.tolist())
display(df.head(3))

assert "path" in df.columns, "manifest_es.csv debe tener columna 'path'"
assert "label" in df.columns, "manifest_es.csv debe tener columna 'label'"

df["label"] = df["label"].astype(str).str.strip().str.lower()

TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

before = len(df)
df = df[df["label"].isin(TARGET_LABELS)].copy()
print("Filtered rows:", before, "->", len(df))

missing = (~df["path"].apply(lambda p: Path(p).exists())).sum()
print("Missing paths:", missing)
if missing:
    print(df.loc[~df["path"].apply(lambda p: Path(p).exists()), "path"].head(20).tolist())
    raise FileNotFoundError("Hay rutas que no existen en manifest_es.csv. Corrige el manifest.")

print("\nLabel counts:")
print(df["label"].value_counts())

present = sorted(df["label"].unique().tolist())
print("\nPresent labels:", present)

LABELS = [l for l in TARGET_LABELS if l in set(present)]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}
df["y"] = df["label"].map(label2id).astype(int)

print("num_labels:", len(LABELS), LABELS)


Rows: 2738
Cols: ['path', 'label', 'dataset', 'speaker', 'split', 'utt_id', 'level', 'lang']


,path,label,dataset,speaker,split,utt_id,level,lang
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_abajo,NaN,NaN
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_adios,NaN,NaN
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_antes,NaN,NaN


Filtered rows: 2738 -> 2738
Missing paths: 0

Label counts:
label
disgust    457
fear       457
sadness    457
anger      456
joy        456
neutral    455
Name: count, dtype: int64

Present labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness']
num_labels: 6 ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness']


## 4) Extracción de características acústicas

Se define una función para extraer características del audio, como MFCC o espectrogramas.

Estas representaciones convierten la señal temporal en una matriz numérica que puede ser procesada por la red neuronal.

In [5]:
train_df, tmp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["y"]
)
val_df, test_df = train_test_split(
    tmp_df, test_size=0.50, random_state=SEED, stratify=tmp_df["y"]
)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("Train label counts:\n", train_df["label"].value_counts())


train: 2190 val: 274 test: 274
Train label counts:
 label
fear       366
anger      365
disgust    365
joy        365
sadness    365
neutral    364
Name: count, dtype: int64


## 5) Definir Dataset personalizado

Creamos una clase `AudioDataset` que:

- Carga el archivo de audio
- Extrae características
- Devuelve el tensor junto con su etiqueta

Esto permite integrar fácilmente los datos en un `DataLoader`.

In [6]:
feature_extractor = ASTFeatureExtractor.from_pretrained(CFG["model_name"])
SR = CFG["sampling_rate"]

time_mask = torchaudio.transforms.TimeMasking(time_mask_param=CFG["time_mask_param"])
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=CFG["freq_mask_param"])

def apply_specaugment_bft(x_bft: torch.Tensor) -> torch.Tensor:
    xs = []
    for x in x_bft:
        x2 = time_mask(x)
        x2 = freq_mask(x2)
        xs.append(x2)
    return torch.stack(xs, dim=0)

class AudioDatasetAST(Dataset):
    def __init__(self, df_: pd.DataFrame):
        self.df = df_.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["path"]
        y = int(row["y"])

        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)

        if sr != SR:
            wav = torchaudio.functional.resample(wav, sr, SR)

        wav = wav / (wav.abs().max() + 1e-9)
        wav = wav.to(torch.float32)

        # evita NaNs raros
        if torch.isnan(wav).any() or torch.isinf(wav).any():
            wav = torch.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)

        return wav, y

# ✅ FIX AQUÍ: convertimos a numpy float32 1D antes de pasar al feature_extractor
def collate_ast(batch):
    waves, ys = zip(*batch)

    waves_np = []
    for w in waves:
        w = w.detach().cpu().contiguous()
        waves_np.append(w.numpy().astype(np.float32))

    inputs = feature_extractor(
        waves_np,
        sampling_rate=SR,
        return_tensors="pt",
        padding=True
    )
    y = torch.tensor(ys, dtype=torch.long)

    # SpecAugment si existe algo 3D tipo (B,T,F) o (B,F,T)
    if CFG["specaugment"]:
        if "input_values" in inputs and inputs["input_values"].dim() == 3:
            xv = inputs["input_values"]
            # suele ser (B,T,128)
            if xv.shape[-1] == 128:
                x_bft = xv.transpose(1, 2)      # (B,128,T)
                x_bft = apply_specaugment_bft(x_bft)
                inputs["input_values"] = x_bft.transpose(1, 2)
        if "input_features" in inputs and inputs["input_features"].dim() == 3:
            xf = inputs["input_features"]
            if xf.shape[-1] == 128:
                x_bft = xf.transpose(1, 2)
                x_bft = apply_specaugment_bft(x_bft)
                inputs["input_features"] = x_bft.transpose(1, 2)

    return inputs, y

train_ds = AudioDatasetAST(train_df)
val_ds   = AudioDatasetAST(val_df)
test_ds  = AudioDatasetAST(test_df)

print("Datasets:", len(train_ds), len(val_ds), len(test_ds))


Datasets: 2190 274 274


## 6) Crear DataLoaders

Se crean los `DataLoader` para entrenamiento y validación.

Estos permiten procesar los datos en batches y optimizar el uso de memoria durante el entrenamiento.

In [7]:
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    persistent_workers=CFG["persistent_workers"],
    collate_fn=collate_ast,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    persistent_workers=CFG["persistent_workers"],
    collate_fn=collate_ast,
)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    persistent_workers=CFG["persistent_workers"],
    collate_fn=collate_ast,
)

print("Loaders:", len(train_loader), len(val_loader), len(test_loader))


Loaders: 92 12 12


## 7) Cargar modelo AST y reutilizar backbone (EN → ES)

Cargamos el modelo **AST (Audio Spectrogram Transformer)** preentrenado y lo adaptamos a nuestro número de clases (`num_labels`).  
Además, incluimos una función para **cargar el backbone** desde un checkpoint entrenado en inglés (si existe), para aprovechar conocimiento previo y acelerar la convergencia en español.

In [8]:
num_labels = len(LABELS)

model = ASTForAudioClassification.from_pretrained(
    CFG["model_name"],
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(DEVICE)

def load_en_backbone(model, ckpt_path: Path):
    if not ckpt_path.exists():
        print("⚠️ No EN ckpt. Using HF pretrained only.")
        return

    ck = torch.load(ckpt_path, map_location="cpu")
    state = ck["model_state"] if isinstance(ck, dict) and "model_state" in ck else ck

    drop = [k for k in state.keys() if k.startswith("classifier.")]
    for k in drop:
        del state[k]

    missing, unexpected = model.load_state_dict(state, strict=False)
    print("✅ EN backbone loaded. Missing:", len(missing), "Unexpected:", len(unexpected))

load_en_backbone(model, CKPT_EN)


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([6])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


✅ EN backbone loaded. Missing: 4 Unexpected: 0


## 8) Balanceo de clases + función de pérdida

Calculamos la distribución de clases en `train_df` y generamos **pesos por clase** para compensar el desbalanceo.  
Estos `class_weights` se incorporan a la función de pérdida (CrossEntropy) para penalizar más los errores en clases minoritarias.

In [9]:
counts = train_df["y"].value_counts().sort_index()
counts = counts.reindex(range(num_labels), fill_value=0).values.astype(np.float32)

weights = (counts.sum() / (counts + 1e-6))
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

print("counts:", counts.tolist())
print("class_weights:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=float(CFG["label_smoothing"])
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(CFG["lr"]),
    weight_decay=float(CFG["weight_decay"])
)

total_steps = math.ceil(len(train_loader) / CFG["grad_accum"]) * CFG["epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

AMP = (DEVICE == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

print("total_steps:", total_steps, "warmup_steps:", warmup_steps, "AMP:", AMP)


counts: [365.0, 365.0, 366.0, 365.0, 364.0, 365.0]
class_weights: [0.99999744 0.99999744 0.99726516 0.99999744 1.0027447  0.99999744]
total_steps: 1150 warmup_steps: 115 AMP: True


## 9) Prueba rápida (1 batch) + AMP (mixed precision)

Antes de entrenar, ejecutamos un **forward/backward** con un batch para comprobar que:
- el modelo recibe correctamente `inputs`
- la pérdida se calcula sin errores
- AMP (mixed precision) funciona con `autocast` + `GradScaler`

Esto evita perder tiempo si hay un fallo de shapes o de GPU.

In [10]:
model.train()
inputs, y = next(iter(train_loader))

y = y.to(DEVICE)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

if DEVICE == "cuda":
    torch.cuda.synchronize()
t0 = time.time()

with torch.amp.autocast("cuda", enabled=AMP):
    out = model(**inputs)
    loss = criterion(out.logits, y)

scaler.scale(loss).backward()

if DEVICE == "cuda":
    torch.cuda.synchronize()
t1 = time.time()

print(f"✅ Sanity OK | fwd+bwd: {t1-t0:.2f}s | loss: {loss.item():.4f}")


✅ Sanity OK | fwd+bwd: 2.30s | loss: 1.9216


## 10) Función de evaluación (VAL/TEST)

Definimos `run_eval(...)` para evaluar el modelo en validación o test sin gradientes.  
Devuelve:
- `loss` medio
- `y_true` y `y_pred` para métricas (accuracy/F1) y matriz de confusión

Esta función se reutiliza durante el entrenamiento para decidir el mejor checkpoint.

In [11]:
@torch.no_grad()
def run_eval(loader, desc="VAL"):
    model.eval()
    losses = []
    ys_all, preds_all = [], []

    pbar = tqdm(loader, desc=desc, leave=True)
    for inputs, y in pbar:
        y = y.to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=AMP):
            out = model(**inputs)
            loss = criterion(out.logits, y)

        losses.append(loss.item())
        preds = out.logits.argmax(dim=1)

        ys_all.append(y.detach().cpu())
        preds_all.append(preds.detach().cpu())
        pbar.set_postfix(loss=float(np.mean(losses)))

    ys_all = torch.cat(ys_all).numpy()
    preds_all = torch.cat(preds_all).numpy()
    return float(np.mean(losses)), ys_all, preds_all

def train_one_epoch(epoch_idx: int):
    model.train()
    running = []
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"TRAIN {epoch_idx:02d}", leave=True)
    for step, (inputs, y) in pbar:
        y = y.to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=AMP):
            out = model(**inputs)
            loss = criterion(out.logits, y) / CFG["grad_accum"]

        scaler.scale(loss).backward()

        if (step + 1) % CFG["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running.append(loss.item() * CFG["grad_accum"])
        lr_now = scheduler.get_last_lr()[0]
        pbar.set_postfix(loss=float(np.mean(running)), lr=float(lr_now))

    return float(np.mean(running))


## 11) Entrenamiento principal (heavy) + checkpoint del mejor modelo

Entrenamos el modelo durante varias épocas y guardamos:
- `BEST_PATH`: el mejor modelo según **val_loss**
- `LAST_PATH`: el último estado al finalizar

Además registramos `history` para analizar la evolución del entrenamiento y detectar sobreajuste.

In [12]:
BEST_PATH = MODELS_DIR / "ast_es_best_heavy.pt"
LAST_PATH = MODELS_DIR / "ast_es_last_heavy.pt"

best_val = float("inf")
bad = 0
history = []

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tr_loss = train_one_epoch(epoch)
    val_loss, y_true, y_pred = run_eval(val_loader, desc=f"VAL {epoch:02d}")
    dt = time.time() - t0

    print(f"Epoch {epoch:02d}/{CFG['epochs']} | train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | dt={dt:.1f}s")
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": val_loss})

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "labels": LABELS,
        "label2id": label2id,
        "id2label": id2label,
        "manifest": str(MANIFEST_ES),
        "cfg": CFG,
    }, LAST_PATH)

    if val_loss < best_val:
        best_val = val_loss
        bad = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "labels": LABELS,
            "label2id": label2id,
            "id2label": id2label,
            "manifest": str(MANIFEST_ES),
            "cfg": CFG,
        }, BEST_PATH)
        print("✅ Guardado BEST:", BEST_PATH)
    else:
        bad += 1
        print(f"early_stop counter {bad}/{CFG['patience']}")
        if bad >= CFG["patience"]:
            print("🛑 Early stopping.")
            break

print("Done. best_val_loss:", best_val)
pd.DataFrame(history).tail()


TRAIN 01:   0%|          | 0/92 [00:00<?, ?it/s]

VAL 01:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 01/25 | train_loss=1.7983 | val_loss=1.6807 | dt=313.0s
✅ Guardado BEST: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\models\ast_es_best_heavy.pt


TRAIN 02:   0%|          | 0/92 [00:00<?, ?it/s]

VAL 02:   0%|          | 0/12 [00:00<?, ?it/s]

...

Epoch 10/25 | train_loss=0.3015 | val_loss=1.3157 | dt=248.9s
early_stop counter 4/5


TRAIN 11:   0%|          | 0/92 [00:00<?, ?it/s]

VAL 11:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 11/25 | train_loss=0.3026 | val_loss=1.3446 | dt=248.8s
early_stop counter 5/5
🛑 Early stopping.
Done. best_val_loss: 1.274732122818629


,epoch,train_loss,val_loss
6,7,0.360256,1.494334
7,8,0.338176,1.427900
8,9,0.318412,1.460114
9,10,0.301532,1.315656
10,11,0.302589,1.344634


## 12) Fine-tuning final (ajuste con LR más bajo)

Cargamos el mejor checkpoint del entrenamiento principal y ajustamos el learning rate (`lr_finetune`).  
En esta fase el objetivo es **afinar** el modelo con cambios más pequeños, mejorando la generalización sin “romper” lo aprendido.

In [13]:
ck = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ck["model_state"], strict=True)
model.to(DEVICE)

for g in optimizer.param_groups:
    g["lr"] = float(CFG["lr_finetune"])

finetune_epochs = 8
total_steps_ft = math.ceil(len(train_loader) / CFG["grad_accum"]) * finetune_epochs
warmup_steps_ft = int(total_steps_ft * 0.10)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps_ft,
    num_training_steps=total_steps_ft
)

print("Finetune epochs:", finetune_epochs, "LR:", CFG["lr_finetune"])

best_val2 = best_val
bad = 0

for epoch in range(1, finetune_epochs + 1):
    t0 = time.time()
    tr_loss = train_one_epoch(epoch)
    val_loss, _, _ = run_eval(val_loader, desc=f"VAL_FT {epoch:02d}")
    dt = time.time() - t0

    print(f"FT Epoch {epoch:02d}/{finetune_epochs} | train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | dt={dt:.1f}s")

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "labels": LABELS,
        "label2id": label2id,
        "id2label": id2label,
        "manifest": str(MANIFEST_ES),
        "cfg": CFG,
    }, LAST_PATH)

    if val_loss < best_val2:
        best_val2 = val_loss
        bad = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "labels": LABELS,
            "label2id": label2id,
            "id2label": id2label,
            "manifest": str(MANIFEST_ES),
            "cfg": CFG,
        }, BEST_PATH)
        print("✅ Guardado BEST (FT):", BEST_PATH)
    else:
        bad += 1
        print(f"FT early_stop counter {bad}/3")
        if bad >= 3:
            print("🛑 FT Early stopping.")
            break

print("Done FT. best_val_loss:", best_val2)


Finetune epochs: 8 LR: 2e-06


TRAIN 01:   0%|          | 0/92 [00:00<?, ?it/s]

VAL_FT 01:   0%|          | 0/12 [00:00<?, ?it/s]

FT Epoch 01/8 | train_loss=0.3542 | val_loss=1.3981 | dt=282.2s
FT early_stop counter 1/3


TRAIN 02:   0%|          | 0/92 [00:00<?, ?it/s]

VAL_FT 02:   0%|          | 0/12 [00:00<?, ?it/s]

FT Epoch 02/8 | train_loss=0.3393 | val_loss=1.3402 | dt=282.4s
FT early_stop counter 2/3


TRAIN 03:   0%|          | 0/92 [00:00<?, ?it/s]

VAL_FT 03:   0%|          | 0/12 [00:00<?, ?it/s]

FT Epoch 03/8 | train_loss=0.3163 | val_loss=1.3427 | dt=281.5s
FT early_stop counter 3/3
🛑 FT Early stopping.
Done FT. best_val_loss: 1.274732122818629


## 13) Evaluación final en TEST + matriz de confusión

Cargamos el mejor modelo guardado y evaluamos en el conjunto de test (nunca visto).  
Mostramos:
- `test_loss`
- `classification_report` por emoción
- `confusion_matrix` para identificar qué clases se confunden entre sí

In [14]:
print("Loading BEST:", BEST_PATH)
ck = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ck["model_state"], strict=True)
model.eval()

test_loss, y_true, y_pred = run_eval(test_loader, desc="TEST")
print("\nTEST loss:", test_loss)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix:\n", cm)


Loading BEST: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\models\ast_es_best_heavy.pt


TEST:   0%|          | 0/12 [00:00<?, ?it/s]


TEST loss: 1.5005039771397908

Classification report:
              precision    recall  f1-score   support

       anger       0.48      0.61      0.54        46
     disgust       0.47      0.37      0.41        46
        fear       0.48      0.36      0.41        45
         joy       0.58      0.57      0.57        46
     neutral       0.52      0.58      0.55        45
     sadness       0.56      0.63      0.59        46

    accuracy                           0.52       274
   macro avg       0.52      0.52      0.51       274
weighted avg       0.52      0.52      0.51       274


Confusion matrix:
 [[28  1  2  8  5  2]
 [ 6 17  5  6  5  7]
 [ 7  6 16  2  4 10]
 [ 9  3  1 26  6  1]
 [ 7  4  2  3 26  3]
 [ 1  5  7  0  4 29]]
